# Project 6: train the three YOLO11n runs on a free Colab GPU

This notebook trains the same three runs as `make train` (frozen backbone baseline, full fine tune,
light augmentation; 12 epochs each at 512 pixels, seed 42) on a GPU, because the 4 CPU cores of the
build environment need about 10 hours. Evaluation, ONNX export and the report run afterwards on CPU
with `make evaluate report`, from the weights this notebook saves.

**Before you run it**

1. In Google Drive, create the folder `MyDrive/rdd2022` and put three files in it:
   `Czech.zip`, `India.zip` (the RDD2022 country zips) and `p06_code.zip` (the code of this project).
2. In Colab: Runtime, Change runtime type, select a GPU (T4 is enough).
3. Runtime, Run all. Colab asks once for access to your Drive.

Each epoch is copied to `MyDrive/rdd2022/backup`. If Colab disconnects, run all cells again: training
resumes from the last finished epoch. The result is `MyDrive/rdd2022/p06_results.zip` (about 20 MB).


In [ ]:
# The GPU that Colab assigned; the notebook stops here when there is none.
import torch
assert torch.cuda.is_available(), 'No GPU: Runtime, Change runtime type, select a GPU'
print(torch.cuda.get_device_name(0), torch.__version__)

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
FOLDER = '/content/drive/MyDrive/rdd2022'
import os

missing = [f for f in ('Czech.zip', 'India.zip', 'p06_code.zip') if not os.path.exists(f'{FOLDER}/{f}')]
assert not missing, f'missing in {FOLDER}: {missing}'

## Code and packages

Colab keeps its own CUDA build of PyTorch; Ultralytics and the other pinned packages of the project are installed beside it.

In [ ]:
%cd /content
!rm -rf work && mkdir work && unzip -q {FOLDER}/p06_code.zip -d work
!pip install -q ultralytics==8.4.184 mlflow==3.17.2 tabulate==0.10.0
!pip install -q --no-deps -e work/common -e work/projects/p06_road_damage
%cd /content/work/projects/p06_road_damage

## Data

The zips are copied to the local disk of the Colab machine, which reads faster than Drive. `road_damage.data` writes the same split as in the build environment (blocks of 25 images, seed 42).

In [ ]:
!mkdir -p data/raw && cp {FOLDER}/Czech.zip {FOLDER}/India.zip data/raw/
!python -m road_damage.data --source data/raw

## Training

About 1 to 2 hours on a T4. The progress bars show one line per epoch.

In [ ]:
!MLFLOW_DISABLE_AGENT_HINT=1 python -m road_damage.train --device 0 --workers 2 --backup {FOLDER}/backup

## Results to Drive

Weights, training histories and the split lists, for evaluation in the build environment. No images are included.

In [ ]:
import zipfile
from pathlib import Path

root = Path('artifacts')
files = [root / 'train_summary.json']
for run in sorted((root / 'runs').iterdir()):
    files += [run / 'weights' / 'best.pt', run / 'results.csv', run / 'args.yaml']
files += sorted(Path('data/yolo').glob('*.txt'))
out = Path(FOLDER) / 'p06_results.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in files:
        archive.write(path, path.as_posix())
print(out, round(out.stat().st_size / 2**20, 1), 'MB,', len(files), 'files')